# 16 - רכבת קלה, חשמלית ומודלים משניים

הניתוח הארצי בנוטבוקים 02-10 הוא, מבחינה מספרית, ניתוח של **אוטובוסים**: 6,796 מתוך 7,792 הקווים ו-412,544 מתוך 420,133 הנסיעות המתוזמנות ב-feed הזה הם `route_type = 3`. כל היתר הוא שגיאת עיגול בסטטיסטיקות המצרפיות, ולכן בלתי נראה בהן. הנוטבוק הזה מוציא את המודלים הקטנים מן הצל הזה ובוחן כל אחד מהם כרשת בפני עצמה:

| `route_type` | התווית שבשימוש כאן | מה זה באמת ב-feed הזה |
|---|---|---|
| 0 | tram/light rail | הקו האדום בירושלים והקו האדום בתל אביב |
| 5 | cable tram | הפוניקולר "כרמלית" בחיפה והרכבלית בחיפה |
| 8 | trolleybus | **לא** טרוליבוסים - מפעילי מוניות שירות (*מונית שירות*) המקודדים תחת סוג זה |
| 715 | demand/other bus | קווי הזנה כפריים מבוססי היענות לביקוש (מטה יהודה, חבל אילות, יואב) |

רכבת (`route_type = 2`) אינה מכוסה כאן **במכוון** - נוטבוק 11 כבר עוסק בה במלואה.

**שאלת המחקר הנדונה כאן:** *האם המודלים המשניים מתנהגים כעותקים מוקטנים של רשת האוטובוסים, או כמשהו שונה מבנית - ומה המשמעות של כך לקריטיות?* המתח המרכזי הוא **עצימות** השירות: לרכבת הקלה יש 8 קווים אך 2,890 נסיעות יומיות (כ-361 נסיעות לקו), ול-cable tram יש 4 קווים עם 3,006 נסיעות (כ-752 נסיעות לקו), לעומת כ-61 נסיעות לקו באוטובוס. מעט מאוד תחנות, עומס גבוה מאוד לתחנה, ו - כפי שמראה החלק המבני - כמעט ללא מסלולים חלופיים. הצירוף הזה הוא בדיוק מה שהופך תחנה לקריטית, וזהו אופן כשל שונה מזה שרשת האוטובוסים מציגה.

מכיוון שלגרפים הללו יש עשרות צמתים ולא עשרות אלפים, **כל מדדי המרכזיות כאן מחושבים במדויק**. נוטבוק 04 נאלץ לקרב betweenness באמצעות Brandes עם דגימת `k` על 30k צמתים; על גרף בן 138 צמתים חישוב Brandes מדויק הוא מיידי, ולכן אין כל סיבה לקבל שגיאת דגימה - ואף לא מתקבלת כזו.

## קלט

* `israel-public-transportation/routes.txt`, `trips.txt`, `agency.txt` - שיוך מודל תחבורה (`route_type` נמצא ב-`routes.txt`; `trips.txt` מקשר בין `trip_id` ל-`route_id`).
* `israel-public-transportation/stop_times.txt` - 816 MB / 15.7M שורות, **אינו מנוהל ב-git**, ומורד מ-Google Drive על ידי אחד התאים להלן ונקרא **בזרימה, שורה אחר שורה**.
* `outputs/nb/01_data_preparation/tables/stops_clean.csv` - שמות תחנות, קואורדינטות, `region`, `metro`. **נדרש**; מיוצר על ידי נוטבוק `01_data_preparation`.
* `outputs/nb/02_graph_construction/tables/nodes.csv` + `edges.csv` - **אופציונלי**; משמש רק לבדיקת ההצלבה הממקמת את המודלים המשניים בתוך הגרף הארצי.
* `outputs/nb/14_*/tables/mode_inventory.csv` - **אופציונלי**; אם נוטבוק 14 הורץ, המלאי שלו מוצלב מול זה המחושב כאן.

## פלט (הכול תחת `outputs/nb/16_lightrail_and_minor_modes/`)

| נתיב | תוכן |
|---|---|
| `tables/lightrail_station_metrics.csv` | שורה אחת לכל תחנה של **כל** מודל משני: `stop_id, stop_name, lat, lon, degree, weighted_degree, mode_label` (בתוספת betweenness מדויק, מספר עצירות, ודגל articulation) |
| `tables/minor_modes_summary.csv` | שורה אחת לכל מודל משני: גודל, צורה, יתירות, עצימות שירות, טווח |
| `lightrail_summary.json` | המספרים המרכזיים לדוח |
| `tables/minor_mode_routes.csv` | כל קו של מודל משני, עם המפעיל ומספר הנסיעות |
| `tables/mode_service_intensity.csv` | נסיעות לקו / עצירות לתחנה עבור **כל ששת** המודלים, כולל אוטובוס ורכבת כקו בסיס |
| `tables/minor_mode_components.csv` | כל רכיב קשירות של כל גרף מודל משני, עם מחלקת הצורה שלו |
| `tables/platform_duplication.csv` | מספר `stop_id` ייחודיים מול מספר `stop_name` ייחודיים לכל מודל (ארטיפקט הרציפים הכיווניים) |
| `tables/minor_mode_single_station_damage.csv` | נזק מדויק מהסרת תחנה בודדת, לכל מודל |
| `tables/hourly_departures.csv` | יציאות לשעת שירות לכל מודל (השעות 24-27 כלולות, ראו ההערה על זמני GTFS) |
| `tables/national_component_modes.csv` | אילו רכיבים בגרף הארצי מאוכלסים על ידי המודלים המשניים (רק אם נוטבוק 02 הורץ) |
| `lightrail_graphs.pkl` | `{mode_label: networkx.Graph}` עבור ארבעת המודלים המשניים |
| `figures/*.png` | קנה מידה, עצימות שירות, גאוגרפיה, betweenness מדויק, פרופיל שעתי, נזק |

דבר מחוץ לתיקייה זו אינו נכתב. בפרט, `outputs/tables`, `outputs/figures` ו-`outputs/rail` (הפלטים הקפואים המצוטטים בדוח) אינם משתנים כלל.

## נוטבוקים שיש להריץ קודם

* **`01_data_preparation`** - נדרש, עבור `stops_clean.csv`.
* `02_graph_construction` - אופציונלי, רק עבור בדיקת ההצלבה של המיקום הארצי.
* `14_mode_inventory` - אופציונלי, רק עבור טענת עקביות.

נוטבוק זה **אינו** תלוי בנוטבוק 15 ואינו עושה שימוש חוזר בגרף האוטובוסים.

## 1. אתחול סביבת העבודה

התא שלהלן הוא אתחול הסביבה הסטנדרטי של הפרויקט, זהה לזה שבנוטבוקים 02 ו-03. הוא מאפשר להריץ את הנוטבוק גם על checkout מקומי וגם על Google Colab: `_ensure(...)` מתקין באמצעות pip רק את החבילות החסרות בפועל (כך שהרצה חוזרת היא זולה), ו-`find_repo_root()` מטפס כלפי מעלה מתיקיית העבודה בחיפוש אחר תיקיית ה-GTFS, ומשכפל את המאגר אל `/content` אם אנו על Colab והוא אינו קיים שם. לאחר מכן הוא קובע את `REPO`, `DATA` ו-`OUT`. כל תא מאוחר יותר תלוי בשלושת הנתיבים הללו, ולכן תא זה חייב לרוץ ראשון.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ספריות, תיקיות השלב ופרמטרי עלות

אנו מייבאים את הערימה המדעית וקובעים את מבנה התיקיות של שלב זה: כל מה שנוצר כאן נכתב אל `outputs/nb/16_lightrail_and_minor_modes/` עם תת-התיקיות `tables/` ו-`figures/`, בהתאם למוסכמה של תיקייה אחת לכל נוטבוק.

הקבועים רוכזו כאן כדי שהבודק יוכל לראות את עלות הנוטבוק במקום אחד:

* `MINOR_ROUTE_TYPES = (0, 5, 8, 715)` - ארבעת המודלים הנבחנים. רכבת (2) שייכת לנוטבוק 11 ואוטובוס (3) לנוטבוק 15.
* `PROGRESS_EVERY = 2_000_000` - הדפסות התקדמות במהלך המעבר הזורם היחיד על `stop_times.txt`. **מעבר זה הוא הצעד היקר היחיד בנוטבוק**: 15.7M שורות, בדרך כלל 20-90 שניות בהתאם למטמון הדיסק (מספר דקות על דיסק Colab קר). כל מה שמגיע אחריו פועל על מאות צמתים לכל היותר והוא מיידי למעשה.
* `EXACT_BETWEENNESS = True` - שימוש ב-betweenness מדויק לפי Brandes. העלות היא `O(n*m)`: עבור המודל המשני הגדול ביותר כאן מדובר בקירוב ב-`153 * 194 ~ 3e4` פעולות, כלומר מיקרו-שניות. דגימה הייתה מהווה אובדן דיוק נטו ללא כל תועלת, ולכן הדגל קיים רק כדי לתעד את הבחירה; יש להשאירו דלוק.
* `FIG_DPI`, `TOP_N`, `MAX_LABELLED_STATIONS` - ענייני תצוגה בלבד.

`MODE_LABELS` עושה שימוש חוזר במחרוזות התוויות המדויקות שנוטבוק 01 כתב אל `route_type_distribution.csv`, כך שטבלאות משלבים שונים מתחברות היטב על `mode_label`.

In [ ]:
# --- Libraries, stage folders and cost knobs ------------------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn')

import csv, json, pickle, time
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)
csv.field_size_limit(10_000_000)   # a few rows of stop_times.txt are unusually long

STAGE = OUT / '16_lightrail_and_minor_modes'
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Cost knobs and mode vocabulary --------------------------------------
MINOR_ROUTE_TYPES = (0, 5, 8, 715)     # tram/light rail, cable tram, trolleybus, demand-responsive
ALL_ROUTE_TYPES = (0, 2, 3, 5, 8, 715)  # used only for the service-intensity baseline
MODE_LABELS = {0: 'tram/light rail', 2: 'rail', 3: 'bus',
               5: 'cable tram', 8: 'trolleybus', 715: 'demand/other bus'}
MODE_SLUGS = {0: 'lightrail', 5: 'cabletram', 8: 'trolleybus', 715: 'demand'}
MODE_COLORS = {0: '#dc2626', 2: '#0f766e', 3: '#94a3b8',
               5: '#0891b2', 8: '#7c3aed', 715: '#ca8a04'}

PROGRESS_EVERY = 2_000_000       # progress print interval for the streaming pass
EXACT_BETWEENNESS = True         # exact Brandes O(n*m) - trivial at this graph size
FIG_DPI = 150                    # figure resolution
TOP_N = 20                       # rows shown in top-N tables
MAX_LABELLED_STATIONS = 80       # safety cap on the labelled light-rail map

print('stage output folder :', STAGE)
print('modes analysed      :', [f'{rt} = {MODE_LABELS[rt]}' for rt in MINOR_ROUTE_TYPES])

## 3. רינדור תוויות בעברית

כל שם תחנה בנוטבוק זה הוא בעברית, ובניגוד למפות הארציות (30k נקודות אנונימיות) מפות המודלים המשניים קטנות דיין כדי לתייג כל תחנה בנפרד - וזה עיקר ערכן. Matplotlib אינו מממש את האלגוריתם הדו-כיווני (bidirectional) של Unicode, ולכן טקסט מימין לשמאל מצויר הפוך. התא שלהלן מבצע monkey-patch חד-פעמי ל-`matplotlib.text.Text.set_text` כך שכל מחרוזת המכילה תווים עבריים מומרת לסדר תצוגה באמצעות `python-bidi` לפני הציור, ובוחר גופן הכולל גליפים עבריים (Arial ב-Windows, DejaVu Sans במקומות אחרים). הפעולה אידמפוטנטית - הרצה חוזרת אינה מערימה patches נוספים. זהו אותו תא שבשימוש בנוטבוקים 02 ו-03.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. איתור שלבים קודמים

תיקיות השלבים מאותרות לפי **קידומת דו-ספרתית** (`OUT.glob('01*')`) ולא לפי slug מדויק, כך ששינוי שם של תיקייה אינו שובר את הנוטבוק. `find_artifact` זורק `FileNotFoundError` המציין את הנוטבוק שיש להריץ, במקום להיכשל מאוחר יותר בשגיאת pandas סתומה.

רק קלט אחד נדרש באמת - `stops_clean.csv` מנוטבוק 01, המספק את שמות התחנות, הקואורדינטות ותוויות `region`/`metro`. הארטיפקטים של נוטבוקים 02 ו-14 נטענים **אם הם קיימים** ומשמשים לבדיקות הצלבה בלבד; היעדרם מוריד את רמת שני חלקים אופציונליים ותו לא.

In [ ]:
# --- Resolve earlier stages by two-digit prefix ---------------------------
def stage_dir(prefix):
    """Return the output folder whose name starts with `prefix` (e.g. '02'), or None."""
    matches = sorted(p for p in OUT.glob(prefix + '*') if p.is_dir())
    return matches[0] if matches else None


def find_artifact(prefix, filename, notebook_hint, required=True):
    """Locate `filename` inside the stage folder `prefix*`; explain how to produce it."""
    sd = stage_dir(prefix)
    if sd is not None:
        direct = sd / filename
        if direct.exists():
            return direct
        matches = sorted(sd.rglob(filename))
        if matches:
            return matches[0]
    if required:
        raise FileNotFoundError(
            f"{filename} was not found under {OUT / (prefix + '*')} - "
            f"run notebook {notebook_hint} first; it writes {filename}."
        )
    return None


stops_path = find_artifact('01', 'stops_clean.csv', '01_data_preparation')
stops_df = pd.read_csv(stops_path, dtype=str, keep_default_na=False, encoding='utf-8-sig')


def _to_float(x):
    """Coordinates arrive as strings and may be blank; return None instead of raising."""
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return v if np.isfinite(v) else None


ATTR = {
    r['stop_id']: {
        'stop_name': r.get('stop_name', '') or '',
        'lat': _to_float(r.get('stop_lat')),
        'lon': _to_float(r.get('stop_lon')),
        'region': r.get('region', '') or '',
        'metro': r.get('metro', '') or '',
        'parent_station': r.get('parent_station', '') or '',
    }
    for r in stops_df.to_dict('records')
}
DEFAULT_ATTR = {'stop_name': '', 'lat': None, 'lon': None,
                'region': '', 'metro': '', 'parent_station': ''}

# Optional cross-check inputs.
nodes_path = find_artifact('02', 'nodes.csv', '02_graph_construction', required=False)
edges_path = find_artifact('02', 'edges.csv', '02_graph_construction', required=False)
inventory14_path = find_artifact('14', 'mode_inventory.csv', '14_mode_inventory', required=False)

print(f'stop attributes : {len(ATTR):,} stops  <-  {stops_path}')
print('optional nodes.csv        :', nodes_path)
print('optional edges.csv        :', edges_path)
print('optional mode_inventory   :', inventory14_path)

## 5. שיוך מודל תחבורה: אילו נסיעות שייכות לאיזה מודל

GTFS מציב את מודל התחבורה על ה-**route**, לא על הנסיעה ובוודאי לא על ה-stop time. כדי לתייג שורת stop-time במודל תחבורה נדרש אפוא צירוף דו-שלבי, ששני שלביו נכנסים בנוחות לזיכרון:

`stop_times.trip_id` -> `trips.route_id` -> `routes.route_type`

`routes.txt` מונה כ-7.8k שורות ו-`trips.txt` כ-420k שורות, ולכן אנו בונים מראש את המילון `trip_mode: trip_id -> route_type` (420k רשומות, כמה עשרות MB). המעבר הזורם מתייג לאחר מכן כל אחת מ-15.7M שורות ה-stop-time בעזרת חיפוש בודד במילון.

כמו כן אנו שומרים את שם המפעיל (agency) לכל קו, משום שהוא זה שהופך את המודלים המשניים למובנים: `route_type = 8` הוא נומינלית "trolleybus" במפרט ה-GTFS, אך המפעילים העומדים מאחורי שמונת הקווים הללו הם חברות מוניות (*מטרו קו*, *מוניות אודליה*, *רב-קווית 4-5*), כלומר מדובר ב**קווי מוניות שירות המקודדים תחת route_type פנוי**, ולא בטרוליבוסים. דיווח עליהם כ-"trolleybus" ללא הסתייגות זו יהיה שגוי, ולכן ההסתייגות נשמרת לאורך כל הטבלאות בנוטבוק זה.

In [ ]:
# --- routes.txt / trips.txt / agency.txt ----------------------------------
for p in (DATA / 'routes.txt', DATA / 'trips.txt', DATA / 'agency.txt'):
    if not p.exists():
        raise FileNotFoundError(f'{p} is missing - the GTFS feed must be present under {DATA}.')

agency_name = {}
with open(DATA / 'agency.txt', encoding='utf-8-sig', newline='') as f:
    for a in csv.DictReader(f):
        agency_name[a.get('agency_id', '')] = a.get('agency_name', '')

route_type_of, route_meta = {}, {}
with open(DATA / 'routes.txt', encoding='utf-8-sig', newline='') as f:
    for r in csv.DictReader(f):
        try:
            rt = int(r['route_type'])
        except (TypeError, ValueError, KeyError):
            continue
        rid = r['route_id']
        route_type_of[rid] = rt
        route_meta[rid] = {
            'route_type': rt,
            'route_short_name': r.get('route_short_name', '') or '',
            'route_long_name': r.get('route_long_name', '') or '',
            'agency': agency_name.get(r.get('agency_id', ''), r.get('agency_id', '')),
        }

trip_mode, trip_route = {}, {}
trips_per_mode, trips_by_route = Counter(), Counter()
routes_with_trips = defaultdict(set)
with open(DATA / 'trips.txt', encoding='utf-8-sig', newline='') as f:
    for t in csv.DictReader(f):
        rt = route_type_of.get(t.get('route_id'))
        if rt is None:
            continue
        tid = t['trip_id']
        trip_mode[tid] = rt
        trips_per_mode[rt] += 1
        trips_by_route[t['route_id']] += 1
        routes_with_trips[rt].add(t['route_id'])
        if rt in MINOR_ROUTE_TYPES:
            trip_route[tid] = t['route_id']

routes_per_mode = Counter(route_type_of.values())
inventory = pd.DataFrame([
    {'route_type': rt,
     'mode_label': MODE_LABELS.get(rt, f'route_type {rt}'),
     'routes': routes_per_mode[rt],
     'routes_with_trips': len(routes_with_trips[rt]),
     'trips': trips_per_mode[rt],
     'trips_per_route': round(trips_per_mode[rt] / max(routes_per_mode[rt], 1), 1),
     'is_minor_mode': rt in MINOR_ROUTE_TYPES}
    for rt in sorted(routes_per_mode)
]).sort_values('trips', ascending=False).reset_index(drop=True)

print(f'routes: {len(route_type_of):,} | trips: {len(trip_mode):,}')
print(f'minor-mode trips to be streamed in detail: {sum(trips_per_mode[rt] for rt in MINOR_ROUTE_TYPES):,}')
inventory

### 5a. רשימת קווי המודלים המשניים, ובדיקת הצלבה אופציונלית מול נוטבוק 14

שלושים וארבעה קווים נושאים את מלוא יקום המודלים המשניים, ולכן ניתן פשוט להדפיס את כולם עם המפעיל ומספר הנסיעות. זוהי בדיקת השפיות הזולה ביותר האפשרית לכך שהצירוף לפי מודל תקין: שורות `route_type = 0` אמורות להיות מסדרונות רכבת קלה מזוהים, שורות `route_type = 5` שתי מערכות הכבלים בחיפה, וכן הלאה. הדבר גם חושף את מוסכמת ה-GTFS שלפיה כל כיוון של קו הוא **`route_id` נפרד** - עובדה שמקבלת חשיבות מבנית שני חלקים להלן.

אם נוטבוק 14 הורץ, `mode_inventory.csv` שלו מושווה לספירות המחושבות כאן; אי-התאמה משמעה ששני הנוטבוקים חלוקים לגבי ה-feed, ויש לחקור זאת ולא למצע בשקט. אם נוטבוק 14 לא הורץ, הבדיקה מדולגת.

In [ ]:
# --- Every minor-mode route, with operator and trip count -----------------
minor_routes = pd.DataFrame([
    {'route_type': meta['route_type'],
     'mode_label': MODE_LABELS.get(meta['route_type'], ''),
     'route_id': rid,
     'agency': meta['agency'],
     'route_short_name': meta['route_short_name'],
     'route_long_name': meta['route_long_name'],
     'trips': trips_by_route.get(rid, 0)}
    for rid, meta in route_meta.items() if meta['route_type'] in MINOR_ROUTE_TYPES
]).sort_values(['route_type', 'trips'], ascending=[True, False]).reset_index(drop=True)
minor_routes.to_csv(TABLES / 'minor_mode_routes.csv', index=False, encoding='utf-8-sig')

print('agencies per minor mode:')
for rt in MINOR_ROUTE_TYPES:
    ags = sorted(set(minor_routes.loc[minor_routes['route_type'] == rt, 'agency']))
    print(f'  {rt:>3} {MODE_LABELS[rt]:<18} {len(ags)} operator(s): {", ".join(ags)}')

# Optional consistency check against notebook 14.
if inventory14_path is not None:
    inv14 = pd.read_csv(inventory14_path, encoding='utf-8-sig')
    merged = inventory.merge(inv14, on='route_type', how='inner', suffixes=('_here', '_nb14'))
    if len(merged):
        bad = merged[(merged['routes_here'] != merged['routes_nb14'])
                     | (merged['trips_here'] != merged['trips_nb14'])]
        print('\nCross-check vs notebook 14 mode_inventory.csv:',
              'consistent' if bad.empty else 'MISMATCH - investigate')
        if not bad.empty:
            display(bad)
else:
    print('\nnotebook 14 output not present - inventory cross-check skipped (not an error).')

minor_routes

## 6. תלות בנתונים חיצוניים: `stop_times.txt`

רצפי העצירות - ולכן גם הקשתות - קיימים רק ב-`stop_times.txt`, שגודלו 816 MB והוא חורג בהרבה ממגבלת גודל הקובץ של GitHub, ולכן הוא **אינו** במאגר. התא שלהלן מוריד אותו מ-Google Drive בהרצה הראשונה, ומדלג על ההורדה אם הקובץ כבר קיים. זוהי תלות הרשת החיצונית היחידה של הנוטבוק; כל השאר נמצא במאגר או מיוצר על ידי נוטבוק 01. ההורדה נמשכת מספר דקות בהרצת Colab ראשונה.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## 7. פענוח זמני שעון של GTFS - מלכודת ה->= 24:00

זמני GTFS **אינם** זמני שעון קיר. הם היסטים מנקודת הייחוס של *יום השירות* (חצות היום פחות שתים עשרה שעות), כך שנסיעה החוצה את חצות נכתבת `24:15:00`, `25:04:00` וכן הלאה, ולא מתגלגלת ל-`00:15:00`. ב-feed הזה כ-1.1% מהשורות נושאות שעה של 24 ומעלה, והרכבת הקלה לבדה תורמת למעלה מאלף מהן.

ההשלכה בוטה: `datetime.strptime(value, '%H:%M:%S')` זורק `ValueError: unconverted data remains` / `hour must be in 0..23` על שורות אלה והיה מקריס את המעבר. לכן איננו בונים `datetime` כלל. `gtfs_seconds` מפענח את שלושת השדות כמספרים שלמים ומחזיר **שניות מאז חצות יום השירות**, ערך שעשוי לעלות על 86,400 - בדיוק הייצוג הרצוי לנו למדידת מרווחי שירות (headways) וטווחי שירות, משום שהוא שומר על כך שיציאה ב-25:04 תבוא *אחרי* יציאה ב-23:50 ולא תיגלגל לתחילת היום.

ה-asserts שלהלן מקבעים את ההתנהגות, כולל מקרה השעות הקטנות ושני המקרים הפגומים.

In [ ]:
# --- GTFS clock -> seconds since service midnight -------------------------
def gtfs_seconds(value):
    """Parse a GTFS 'HH:MM:SS' field into seconds since service midnight.

    Hours >= 24 are legal in GTFS ('25:30:00' is 01:30 on the next calendar day) and
    are preserved as > 86400 seconds. datetime/strptime must never be used here - it
    raises on hour >= 24, which is ~1.1% of the rows in this feed.
    """
    if not value:
        return None
    parts = value.strip().split(':')
    if len(parts) != 3:
        return None
    try:
        h, m, s = int(parts[0]), int(parts[1]), int(parts[2])
    except ValueError:
        return None
    return h * 3600 + m * 60 + s


def hhmm(sec):
    """Format seconds-since-service-midnight as HH:MM, keeping hours >= 24 visible."""
    if sec is None:
        return ''
    return f'{int(sec) // 3600:02d}:{int(sec) % 3600 // 60:02d}'


assert gtfs_seconds('05:10:00') == 5 * 3600 + 10 * 60
assert gtfs_seconds('25:30:00') == 25 * 3600 + 30 * 60 == 91800
assert gtfs_seconds('24:00:00') == 86400
assert gtfs_seconds('') is None and gtfs_seconds('not a time') is None
assert hhmm(91800) == '25:30'
print('gtfs_seconds OK -  "25:30:00" ->', gtfs_seconds('25:30:00'),
      'seconds since service midnight, formatted as', hhmm(gtfs_seconds('25:30:00')))

## 8. המעבר הזורם על `stop_times.txt`

זהו התא היקר היחיד. הקובץ נקרא באמצעות `csv.reader` שורה אחת בכל פעם - הוא לעולם אינו נטען כולו ל-DataFrame - וכל שורה מתויגת במודל התחבורה שלה באמצעות מילון `trip_mode` שנבנה בחלק 5. לאחר מכן נרשמים שני דברים שונים:

1. **עבור כל מודל, כולל אוטובוס ורכבת**: מספר עצירות התחנה (stop calls) וקבוצת התחנות הייחודיות המשורתות. שני מונים זולים אלה הם שמאפשרים לחלק 10 להשוות את המודלים המשניים לקו בסיס של אוטובוס בלי מעבר נוסף על הקובץ.
2. **עבור ארבעת המודלים המשניים בלבד**: הפירוט המלא של כל עצירה - `(stop_sequence, stop_id, departure_seconds)` מקובץ לפי `trip_id`. מדובר בכ-88,600 שורות בלבד מתוך 15.7M (המודלים המשניים הם 0.6% מה-feed), כך ששמירתן בזיכרון עולה כמה MB וקונה לנו רצפי עצירות מדויקים וזמני יציאה מדויקים.

שימו לב להבדל מנוטבוק 02, שנאלץ לבנות קשתות *אינקרמנטלית* משום שלא יכול היה להחזיק 15.7M שורות. כאן אנו יכולים להרשות לעצמנו לצבור את שורות המודלים המשניים ולמיין כל נסיעה לפי `stop_sequence` בדיעבד, ולכן נוטבוק זה **אינו** מסתמך על כך שה-feed ממוין לפי `(trip_id, stop_sequence)` - הוא מסתמך רק על כך ש-`stop_sequence` נכון בתוך כל נסיעה. זו הנחה חלשה יותר מזו שנוטבוק 02 מניח.

שורות שה-`trip_id` שלהן אינו קיים ב-`trips.txt` נספרות ומדולגות, ולא מושמטות בשקט.

In [ ]:
# --- Single streaming pass over 15.7M rows --------------------------------
def stream_stop_times(path, trip_mode, minor_types, progress_every=PROGRESS_EVERY):
    """One pass over stop_times.txt.

    Returns:
      calls       : {trip_id: [(stop_sequence, stop_id, departure_seconds), ...]} - MINOR modes only
      mode_calls  : {route_type: number of stop-time rows}          - all modes
      mode_stops  : {route_type: set of stop_ids}                   - all modes
      stats       : counters describing the pass
    """
    minor = set(minor_types)
    calls = defaultdict(list)
    mode_calls = Counter()
    mode_stops = defaultdict(set)
    rows_read = unknown_trip_rows = bad_time_rows = bad_seq_rows = 0
    t0 = time.time()

    with open(path, encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f)
        header = next(reader)
        ti = header.index('trip_id')
        si = header.index('stop_id')
        qi = header.index('stop_sequence')
        di = header.index('departure_time')
        for row in reader:
            rows_read += 1
            trip = row[ti]
            mode = trip_mode.get(trip)
            if mode is None:
                unknown_trip_rows += 1
            else:
                stop = row[si]
                mode_calls[mode] += 1
                mode_stops[mode].add(stop)
                if mode in minor:
                    try:
                        seq = int(row[qi])
                    except (ValueError, IndexError):
                        seq = len(calls[trip])   # fall back to file order
                        bad_seq_rows += 1
                    dep = gtfs_seconds(row[di]) if di < len(row) else None
                    if dep is None:
                        bad_time_rows += 1
                    calls[trip].append((seq, stop, dep))
            if progress_every and rows_read % progress_every == 0:
                kept = sum(len(v) for v in calls.values())
                print(f'    {rows_read:,} rows | {kept:,} minor-mode calls kept '
                      f'| {time.time() - t0:,.0f}s')

    stats = {
        'stop_times_rows': rows_read,
        'rows_with_unknown_trip': unknown_trip_rows,
        'minor_mode_trips': len(calls),
        'minor_mode_calls': sum(len(v) for v in calls.values()),
        'unparsable_departure_times': bad_time_rows,
        'unparsable_stop_sequences': bad_seq_rows,
        'elapsed_seconds': round(time.time() - t0, 1),
    }
    return calls, mode_calls, mode_stops, stats


print(f'Streaming {STOP_TIMES.name} (15.7M rows) - this is the only slow cell ...')
calls, mode_calls, mode_stops, stream_stats = stream_stop_times(
    STOP_TIMES, trip_mode, MINOR_ROUTE_TYPES)

print('\nPass statistics:')
for k, v in stream_stats.items():
    print(f'  {k}: {v:,}' if isinstance(v, int) else f'  {k}: {v}')
print('\nStop calls and distinct stops per mode (whole feed):')
for rt in sorted(mode_calls, key=lambda r: -mode_calls[r]):
    print(f'  {rt:>3} {MODE_LABELS.get(rt, "?"):<18} calls={mode_calls[rt]:>12,}  stops={len(mode_stops[rt]):>7,}')

## 9. בניית גרף אחד לכל מודל משני

המודל הוא בדיוק גרף סמיכות הנסיעות (trip-adjacency graph) שהוגדר בנוטבוק 02, מוגבל למודל תחבורה יחיד: צומת הוא תחנה המשורתת על ידי אותו מודל, וקשת מכוונת `u -> v` קיימת כאשר נסיעה כלשהי של אותו מודל עוצרת ב-`v` מיד לאחר `u`, במשקל מספר הנסיעות מסוג זה. ההיטל הלא-מכוון מסכם את שני הכיוונים, כך שמשקל לא-מכוון שומר על משמעותו כ"מספר השירותים החוצים את הקשת הזו בכל אחד מהכיוונים".

הגבלה למודל אחד אינה זהה ללקיחת תת-הגרף המושרה של הגרף הארצי על תחנות אותו מודל: מקטע אוטובוס בין שתי תחנות רכבת קלה יופיע בתת-הגרף המושרה אך אינו קשת של רכבת קלה. אנו בונים מתוך הנסיעות של המודל עצמו, וזוהי הבנייה הנכונה.

לצד הגרפים אנו שומרים, לכל מודל ולכל תחנה, את מספר עצירות התחנה, את קבוצת הקווים המשרתים אותה, ואת רשימת זמני היציאה - חומר הגלם לחלקים העוסקים בעצימות ובמרווחי השירות.

In [ ]:
# --- Per-mode trip-adjacency graphs ---------------------------------------
def build_mode_structures(calls, trip_mode, trip_route, minor_types, attr):
    """Turn buffered stop calls into one directed + one undirected graph per mode."""
    edge_count = {rt: defaultdict(int) for rt in minor_types}
    stop_calls = {rt: Counter() for rt in minor_types}
    stop_routes = {rt: defaultdict(set) for rt in minor_types}
    stop_departures = {rt: defaultdict(list) for rt in minor_types}
    trips_seen = Counter()
    self_loops = Counter()
    one_stop_trips = Counter()

    for trip_id, rows in calls.items():
        rt = trip_mode[trip_id]
        route = trip_route.get(trip_id, '')
        trips_seen[rt] += 1
        ordered = sorted(rows, key=lambda r: r[0])
        if len(ordered) < 2:
            one_stop_trips[rt] += 1
        for _, stop, dep in ordered:
            stop_calls[rt][stop] += 1
            stop_routes[rt][stop].add(route)
            if dep is not None:
                stop_departures[rt][stop].append(dep)
        seq = [stop for _, stop, _ in ordered]
        for u, v in zip(seq, seq[1:]):
            if u == v:
                self_loops[rt] += 1        # same stop twice in a row: no connectivity info
            else:
                edge_count[rt][(u, v)] += 1

    graphs = {}
    for rt in minor_types:
        D = nx.DiGraph()
        for (u, v), c in edge_count[rt].items():
            D.add_edge(u, v, weight=c)
        G = nx.Graph()
        for u, v, data in D.edges(data=True):
            if G.has_edge(u, v):
                G[u][v]['weight'] += data['weight']
            else:
                G.add_edge(u, v, weight=data['weight'])
        for graph in (G, D):
            for n in graph.nodes():
                graph.nodes[n].update(attr.get(n, DEFAULT_ATTR))
        graphs[rt] = {'G': G, 'D': D}
    return graphs, stop_calls, stop_routes, stop_departures, trips_seen, self_loops, one_stop_trips


(graphs, stop_calls, stop_routes, stop_departures,
 trips_seen, self_loops, one_stop_trips) = build_mode_structures(
    calls, trip_mode, trip_route, MINOR_ROUTE_TYPES, ATTR)

for rt in MINOR_ROUTE_TYPES:
    G, D = graphs[rt]['G'], graphs[rt]['D']
    served = len(stop_calls[rt])
    print(f'{MODE_LABELS[rt]:<18} trips={trips_seen[rt]:>5,} | stops served={served:>4} | '
          f'graph nodes={G.number_of_nodes():>4} | directed edges={D.number_of_edges():>4} | '
          f'undirected edges={G.number_of_edges():>4} | self-loops skipped={self_loops[rt]}')
    if served != G.number_of_nodes():
        print(f'    note: {served - G.number_of_nodes()} stop(s) are served but have no '
              f'neighbour (single-stop trips: {one_stop_trips[rt]}) and are therefore not nodes')

## 10. עצימות שירות: תמצית הנוטבוק הזה

כאן המודלים המשניים חדלים להיראות משניים. שני יחסים מחושבים עבור **כל ששת** המודלים, כך שאוטובוס ורכבת משמשים קו בסיס:

* **נסיעות לקו** = נסיעות מתוזמנות / קווים. עד כמה כל קו מנוצל.
* **עצירות לתחנה** = שורות stop-time / תחנות ייחודיות משורתות. כמה שירות רואה התחנה הממוצעת של אותו מודל ביום שירות.

שני המדדים הם מדדי צד-היצע. GTFS אינו נושא נתוני נסועה (ridership), ולכן דבר כאן אינו אומר כמה *אנשים* משתמשים בשירותים הללו - רק כמה שירות מתוזמן. מגבלה זו נגררת לכל טענת קריטיות בנוטבוק זה ומצוינת שוב במסקנות.

עיוות אחד שיש לשים אליו לב בפלט, בכנות: *עצירות לתחנה* עבור רכבת קלה אינו גבוה דרמטית מהממוצע של האוטובוס, והסיבה היא ארטיפקט נתונים ולא עובדה על חשמליות - ה-feed מקצה לכל כיוון של קו רכבת קלה `stop_id` נפרד של רציף, כך שהשירות של תחנה פיזית אחת מפוצל בין שתי שורות. חלק 11 מודד את הכפילות הזו במפורש; הנתון ל-**תחנה** גדול בערך פי שניים מהנתון ל-`stop_id`. *נסיעות לקו* אינו מושפע מהארטיפקט והוא ההשוואה הנקייה יותר.

In [ ]:
# --- Service intensity for every mode -------------------------------------
intensity_rows = []
for rt in sorted(set(list(ALL_ROUTE_TYPES) + list(mode_calls))):
    stops_n = len(mode_stops.get(rt, ()))
    routes_n = routes_per_mode.get(rt, 0)
    trips_n = trips_per_mode.get(rt, 0)
    intensity_rows.append({
        'route_type': rt,
        'mode_label': MODE_LABELS.get(rt, f'route_type {rt}'),
        'routes': routes_n,
        'trips': trips_n,
        'stops': stops_n,
        'stop_calls': mode_calls.get(rt, 0),
        'trips_per_route': round(trips_n / routes_n, 1) if routes_n else np.nan,
        'calls_per_stop': round(mode_calls.get(rt, 0) / stops_n, 1) if stops_n else np.nan,
        'stops_per_route': round(stops_n / routes_n, 1) if routes_n else np.nan,
        'is_minor_mode': rt in MINOR_ROUTE_TYPES,
    })
intensity = pd.DataFrame(intensity_rows).sort_values('trips_per_route', ascending=False)
intensity.to_csv(TABLES / 'mode_service_intensity.csv', index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
panels = [('trips', 'Scheduled trips per mode (log)'),
          ('trips_per_route', 'Trips per route - service intensity'),
          ('calls_per_stop', 'Stop calls per stop_id')]
for ax, (col, title) in zip(axes, panels):
    d = intensity.sort_values(col, ascending=True)
    colors = [MODE_COLORS.get(rt, '#94a3b8') for rt in d['route_type']]
    bars = ax.barh(range(len(d)), d[col].to_numpy(), color=colors)
    ax.set_yticks(range(len(d)))
    ax.set_yticklabels(d['mode_label'], fontsize=9)
    ax.set_title(title, fontsize=11)
    if col == 'trips':
        ax.set_xscale('log')
    for bar, val in zip(bars, d[col].to_numpy()):
        ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
                f'  {val:,.0f}', va='center', fontsize=8)
    ax.margins(x=0.18)
fig.suptitle('Minor modes are few routes worked very hard (bus / rail shown for scale)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / 'service_intensity.png', dpi=FIG_DPI)
plt.show()

intensity

## 11. רציפים מול תחנות - ארטיפקט נתונים המשנה את הטופולוגיה

לפני מדידת הצורה עלינו להיות מפורשים לגבי מהו *צומת* עבור מודלים אלה. ב-feed הזה קו רכבת קלה מפורסם כשני `route_id` (אחד לכל כיוון), ולכל כיוון יש `stop_id` **משלו** של רציף, כאשר שני הרציפים של תחנה פיזית אחת נושאים אותו `stop_name`. ה-cable tram, לעומת זאת, משתמש ב-`stop_id` יחיד לכל תחנה עבור שני הכיוונים.

ההשלכה היא מבנית, לא קוסמטית: מכיוון שרציף כיוון-העלייה ורציף כיוון-הירידה לעולם אינם מופיעים באותה נסיעה, אין קשת המחברת ביניהם, וקו רכבת קלה פיזי יחיד מופיע בגרף כ**שני מסדרונות חד-כיווניים זרים**. לפיכך כל טענה מהצורה "לרשת הרכבת הקלה יש N רכיבים" היא בחלקה טענה על מוסכמות הפרסום של GTFS.

התא שלהלן מכמת את הכפילות - `stop_id` ייחודיים מול `stop_name` ייחודיים לכל מודל - כך שכל ספירה מאוחרת יותר תיקרא עם המקדם הנכון בחשבון. אנו **נמנעים במכוון** ממיזוג רציפים לפי שם: התנגשויות שמות בין תחנות שאינן קשורות במקומות אחרים בארץ הופכות מיזוג לפי שם ללא בטוח, והגרף שאנו מנתחים חייב להישאר אותו אובייקט שבו משתמש שאר הפרויקט.

In [ ]:
# --- Platform duplication: stop_ids vs physical station names -------------
dup_rows = []
for rt in MINOR_ROUTE_TYPES:
    ids = set(stop_calls[rt])
    names = {ATTR.get(s, DEFAULT_ATTR)['stop_name'] for s in ids}
    with_parent = sum(1 for s in ids if ATTR.get(s, DEFAULT_ATTR)['parent_station'])
    dup_rows.append({
        'route_type': rt,
        'mode_label': MODE_LABELS[rt],
        'stop_ids': len(ids),
        'distinct_stop_names': len(names),
        'stop_ids_per_name': round(len(ids) / max(len(names), 1), 2),
        'stop_ids_with_parent_station': with_parent,
    })
platform_dup = pd.DataFrame(dup_rows)
platform_dup.to_csv(TABLES / 'platform_duplication.csv', index=False, encoding='utf-8-sig')

for r in platform_dup.itertuples():
    verdict = ('directional platforms are separate stop_ids'
               if r.stop_ids_per_name > 1.5 else 'one stop_id per physical station')
    print(f'{r.mode_label:<18} {r.stop_ids:>4} stop_ids / {r.distinct_stop_names:>4} names '
          f'= {r.stop_ids_per_name:.2f}  ->  {verdict}')
platform_dup

## 12. גודל, צורה ויתירות של כל גרף מודל

כעת לאפיון המבני. עבור כל מודל אנו מחשבים, במדויק:

* **גודל**: צמתים, קשתות מכוונות ולא-מכוונות, מספר רכיבי קשירות, חלקו של הרכיב הגדול ביותר, דרגה ממוצעת ומרבית.
* **יתירות**, באמצעות **המספר הציקלומטי** (circuit rank) `mu = m - n + c`, מספר המעגלים הבלתי תלויים בגרף בעל `n` צמתים, `m` קשתות לא-מכוונות ו-`c` רכיבים. `mu = 0` משמעו שהגרף הוא יער: **בין כל שתי תחנות קיים בדיוק מסלול אחד, וכל קשת פנימית היא bridge**. `mu > 0` משמעו שקיים לפחות מסלול חלופי אחד היכן שהוא. עבור רשתות תחבורה זהו מדד היתירות היחיד הנקי ביותר.
* **מחלקת צורה** לכל רכיב, מתוך סדרת הדרגות:
  * `isolated` - צומת בודד;
  * `path` - עץ שדרגתו המרבית <= 2 (קו);
  * `cycle` - כל צומת בדרגה 2 (לולאה);
  * `star` - עץ שבו צומת אחד סמוך לכל האחרים (מסוף מסוג hub-and-spoke);
  * `tree` - חסר מעגלים, עם הסתעפות אך ללא לולאות;
  * `meshed` - מכיל מעגל אחד לפחות.
* **נקודות חיתוך (articulation points)** ו-**גשרים (bridges)** - נקודות הכשל הבודדות המדויקות, באמצעות אותן פרוצדורות בזמן לינארי שבנוטבוק 03.
* **Betweenness מדויק** (Brandes, מנורמל, ללא משקלים). נוטבוק 04 נאלץ לדגום על הגרף הארצי; כאן `n` הוא מאות בודדות לכל היותר, ולכן ערכים מדויקים מחושבים במילישניות ואין שגיאת דגימה שיש להסתייג ממנה.
* **טווח שירות ומרווח שירות (headway)**: היציאה הראשונה והאחרונה בשניות מאז חצות יום השירות (כך שיציאה ב-25:04 ממוינת אחרי 23:50), והחציון של הפער בין יציאות עוקבות בתחנה העמוסה ביותר של המודל - קריאה ישירה של תדירות השירות.

הקוטר (diameter) מדווח עבור הרכיב הגדול ביותר בלבד, ורק כאשר הוא קשיר ומכיל יותר מצומת אחד.

In [ ]:
# --- Structural characterisation of each minor mode -----------------------
def classify_component(H):
    """Name the shape of one connected component from its degree sequence."""
    n, m = H.number_of_nodes(), H.number_of_edges()
    if n == 1:
        return 'isolated'
    degs = sorted(d for _, d in H.degree())
    acyclic = (m == n - 1)
    if acyclic:
        if degs[-1] <= 2:
            return 'path'
        if degs[-1] == n - 1:
            return 'star'
        return 'tree'
    if all(d == 2 for d in degs):
        return 'cycle'
    return 'meshed'


def headway_minutes(departure_lists):
    """Median gap between consecutive departures at the busiest stop, in minutes."""
    if not departure_lists:
        return np.nan
    busiest = max(departure_lists.values(), key=len)
    if len(busiest) < 2:
        return np.nan
    d = np.diff(np.sort(np.asarray(busiest, dtype=float)))
    d = d[d > 0]
    return round(float(np.median(d)) / 60.0, 1) if len(d) else np.nan


BETW = {}
summary_rows, component_rows = [], []
for rt in MINOR_ROUTE_TYPES:
    G, D = graphs[rt]['G'], graphs[rt]['D']
    n, m = G.number_of_nodes(), G.number_of_edges()
    comps = sorted(nx.connected_components(G), key=len, reverse=True)
    degs = np.array([d for _, d in G.degree()]) if n else np.array([0])
    ap = sorted(set(nx.articulation_points(G))) if n else []
    br = list(nx.bridges(G)) if n else []
    mu = m - n + len(comps) if n else 0
    BETW[rt] = nx.betweenness_centrality(G, normalized=True) if (n and EXACT_BETWEENNESS) else {}

    shapes = []
    for i, c in enumerate(comps):
        H = G.subgraph(c)
        shape = classify_component(H)
        shapes.append(shape)
        try:
            diam = nx.diameter(H) if len(c) > 1 else 0
        except (nx.NetworkXError, nx.NetworkXNoPath):
            diam = np.nan
        sample = sorted((G.nodes[x]['stop_name'] or x) for x in c)[:3]
        component_rows.append({
            'route_type': rt, 'mode_label': MODE_LABELS[rt], 'component_id': i,
            'nodes': len(c), 'edges': H.number_of_edges(), 'shape': shape,
            'cyclomatic_number': H.number_of_edges() - len(c) + 1,
            'diameter': diam,
            'metro': Counter(G.nodes[x]['metro'] for x in c).most_common(1)[0][0],
            'region': Counter(G.nodes[x]['region'] for x in c).most_common(1)[0][0],
            'sample_stations': ' | '.join(sample),
        })

    deps_all = [d for lst in stop_departures[rt].values() for d in lst]
    calls_total = mode_calls.get(rt, 0)
    stops_total = len(stop_calls[rt])
    summary_rows.append({
        'route_type': rt,
        'mode_label': MODE_LABELS[rt],
        'routes': routes_per_mode.get(rt, 0),
        'trips': trips_per_mode.get(rt, 0),
        'stops': stops_total,
        'graph_nodes': n,
        'directed_edges': D.number_of_edges(),
        'undirected_edges': m,
        'components': len(comps),
        'largest_component_nodes': len(comps[0]) if comps else 0,
        'largest_component_share': round(len(comps[0]) / n, 4) if n else np.nan,
        'avg_degree': round(float(degs.mean()), 2),
        'max_degree': int(degs.max()),
        'density': round(nx.density(G), 5) if n > 1 else np.nan,
        'cyclomatic_number': mu,
        'edges_per_node': round(m / n, 3) if n else np.nan,
        'articulation_points': len(ap),
        'articulation_point_share': round(len(ap) / n, 4) if n else np.nan,
        'bridges': len(br),
        'bridge_share': round(len(br) / m, 4) if m else np.nan,
        'shape_class': ', '.join(f'{v}x {k}' for k, v in Counter(shapes).most_common()),
        'diameter_largest_component': component_rows[-len(comps)]['diameter'] if comps else np.nan,
        'trips_per_route': round(trips_per_mode.get(rt, 0) / routes_per_mode.get(rt, 1), 1),
        'stop_calls': calls_total,
        'calls_per_stop': round(calls_total / stops_total, 1) if stops_total else np.nan,
        'max_calls_at_a_stop': max(stop_calls[rt].values()) if stops_total else 0,
        'first_departure': hhmm(min(deps_all)) if deps_all else '',
        'last_departure': hhmm(max(deps_all)) if deps_all else '',
        'service_span_hours': round((max(deps_all) - min(deps_all)) / 3600, 1) if deps_all else np.nan,
        'departures_after_midnight': int(sum(1 for d in deps_all if d >= 86400)),
        'median_headway_busiest_stop_minutes': headway_minutes(stop_departures[rt]),
        'max_exact_betweenness': round(max(BETW[rt].values()), 4) if BETW[rt] else np.nan,
        'exact_betweenness': bool(EXACT_BETWEENNESS),
    })

minor_summary = pd.DataFrame(summary_rows)
minor_summary.to_csv(TABLES / 'minor_modes_summary.csv', index=False, encoding='utf-8-sig')
components_df = pd.DataFrame(component_rows)
components_df.to_csv(TABLES / 'minor_mode_components.csv', index=False, encoding='utf-8-sig')

print('saved:', TABLES / 'minor_modes_summary.csv')
print('saved:', TABLES / 'minor_mode_components.csv')
display(minor_summary[['mode_label', 'graph_nodes', 'undirected_edges', 'components',
                       'avg_degree', 'max_degree', 'cyclomatic_number', 'shape_class',
                       'articulation_points', 'bridges']])
components_df

### 12a. קריאת מספרי היתירות

התא שלהלן הופך את שתי העמודות המבניות למשפט שמשמעותי לקריטיות. מודל שמספרו הציקלומטי אפס הוא **יער**: כל קשת היא bridge, כל צומת שאינו עלה הוא articulation point, וקיים בדיוק מסלול אחד בין כל שתי תחנות שהוא מסוגל לשרת. בגרף כזה לשאלה "איזו תחנה קריטית?" יש תשובה מנוונת - *כמעט כולן* - והדירוג המשמעותי היחיד הוא לפי **כמה** תנועה תיוותר מנותקת מכל חתך, וזה בדיוק מה שחלק 16 מודד.

זהו ההבדל המבני מרשת האוטובוסים, שבה נוטבוק 03 מצא articulation points בכ-3% מהתחנות בלבד.

In [ ]:
# --- Redundancy verdict per mode ------------------------------------------
for r in minor_summary.itertuples():
    if r.graph_nodes == 0:
        print(f'{r.mode_label:<18} no graph (no multi-stop trips)')
        continue
    if r.cyclomatic_number == 0:
        verdict = ('FOREST - zero independent cycles: exactly one path between any two '
                   'stations, every link is a bridge')
    else:
        verdict = (f'{r.cyclomatic_number} independent cycle(s): some alternative paths exist')
    print(f'{r.mode_label:<18} n={r.graph_nodes:<4} m={r.undirected_edges:<4} '
          f'mu={r.cyclomatic_number:<3} AP={r.articulation_points} '
          f'({r.articulation_point_share:.0%} of stations)  bridges={r.bridges} '
          f'({r.bridge_share:.0%} of links)')
    print(f'{"":<18} -> {verdict}')

print()
print('For contrast, the national (mostly bus) graph in notebook 03 has articulation points')
print('at roughly 3% of its stations and bridges on under 2% of its links.')

## 13. מדדים ברמת התחנה, עם betweenness מדויק

תא זה כותב את טבלת החוזה `tables/lightrail_station_metrics.csv`. היא מכילה **שורה אחת לכל תחנה של כל מודל משני**, לא רק רכבת קלה - עמודת `mode_label` היא המפרידה ביניהם, וצרכן המעוניין ברכבת קלה בלבד צריך לסנן על `mode_label == 'tram/light rail'`. העמודות הנדרשות מופיעות ראשונות ושמן בדיוק כפי שחוזה הנתונים מגדיר:

`stop_id, stop_name, lat, lon, degree, weighted_degree, mode_label`

`weighted_degree` הוא סכום משקלי הקשתות הלא-מכוונות בתחנה, כלומר **מספר השירותים המתוזמנים החוצים את התחנה ביום שירות, בספירת שני הכיוונים** - מדד העומס הטבעי למודל זה.

עמודות נוספות מצורפות לצורכי הניתוח כאן וניתן להתעלם מהן בבטחה: betweenness מדויק, עצירות תחנה, מספר הקווים המשרתים את התחנה, דגל ה-articulation point, מזהה הרכיב, ו-`region`/`metro`. התחנות ממוינות לפי מודל ולאחר מכן לפי betweenness מדויק בסדר יורד.

In [ ]:
# --- Station metrics for every minor mode ---------------------------------
station_rows = []
for rt in MINOR_ROUTE_TYPES:
    G = graphs[rt]['G']
    if G.number_of_nodes() == 0:
        continue
    comp_of = {node: i for i, c in enumerate(
        sorted(nx.connected_components(G), key=len, reverse=True)) for node in c}
    ap = set(nx.articulation_points(G))
    wdeg = dict(G.degree(weight='weight'))
    for node, data in G.nodes(data=True):
        station_rows.append({
            'stop_id': node,
            'stop_name': data.get('stop_name', ''),
            'lat': data.get('lat'),
            'lon': data.get('lon'),
            'degree': G.degree(node),
            'weighted_degree': int(wdeg.get(node, 0)),
            'mode_label': MODE_LABELS[rt],
            'route_type': rt,
            'exact_betweenness': round(BETW[rt].get(node, 0.0), 6),
            'stop_calls': int(stop_calls[rt].get(node, 0)),
            'n_routes': len(stop_routes[rt].get(node, ())),
            'is_articulation_point': node in ap,
            'component_id': comp_of.get(node, -1),
            'region': data.get('region', ''),
            'metro': data.get('metro', ''),
        })

station_metrics = (pd.DataFrame(station_rows)
                   .sort_values(['route_type', 'exact_betweenness', 'weighted_degree'],
                                ascending=[True, False, False])
                   .reset_index(drop=True))
station_metrics.to_csv(TABLES / 'lightrail_station_metrics.csv', index=False, encoding='utf-8-sig')
print('saved:', TABLES / 'lightrail_station_metrics.csv',
      f'({len(station_metrics):,} stations across {station_metrics["mode_label"].nunique()} modes)')

missing_coords = station_metrics[['lat', 'lon']].isna().any(axis=1).sum()
print('stations with missing coordinates:', int(missing_coords))

print('\nTop light-rail stations by exact betweenness:')
display(station_metrics[station_metrics['route_type'] == 0]
        .head(TOP_N)[['stop_name', 'metro', 'degree', 'weighted_degree',
                      'exact_betweenness', 'stop_calls', 'is_articulation_point']])
print('Busiest stations of every minor mode (by services crossing the station):')
station_metrics.sort_values('weighted_degree', ascending=False).head(TOP_N)[
    ['mode_label', 'stop_name', 'metro', 'degree', 'weighted_degree', 'stop_calls']]

## 14. הגאוגרפיה של כל מודל

ארבעה פאנלים, אחד לכל מודל, כל אחד מצייר את הקשתות בפועל כקטעי קו בין קואורדינטות התחנות, ואת התחנות כנקודות שגודלן לפי מספר השירותים החוצים אותן. זהו פיזור קווי אורך/רוחב פשוט ולא מפה מוטלת, ולכן יחס הממדים של כל פאנל נקבע ל-`1 / cos(mean latitude)` כדי לשמור על מרחקים מדויקים בקירוב.

הפאנלים עונים על החצי הגאוגרפי של השאלה: היכן המודלים הללו קיימים בפועל? התשובה הקצרה הנראית בגרפים היא שכל אחד מהם הוא *ארטיפקט מטרופוליני יחיד* - רכבת קלה בירושלים ובתל אביב, cable tram בחיפה, מוניות שירות בתוך תל אביב, ושירותי ההיענות לביקוש באשכולות כפריים (שפלת יהודה ואזור אילות בדרום הרחוק). אף אחד מהם אינו רשת ארצית, וזו בדיוק הסיבה לכך שהמצרפים הארציים אינם רואים אותם.

תחנות ללא קואורדינטות שמישות מוחרגות ונספרות; הבדיקה היא לקיומו של ערך סופי, כך שקואורדינטה של `0.0` בדיוק תישמר בעוד ש-`NaN` תושמט.

In [ ]:
# --- One geographic panel per mode ----------------------------------------
def has_coords(data):
    """True only when both coordinates are present and finite (0.0 included)."""
    lat, lon = data.get('lat'), data.get('lon')
    return (lat is not None and lon is not None
            and np.isfinite(lat) and np.isfinite(lon))


def draw_mode(ax, rt, label_stations=False, max_labels=MAX_LABELLED_STATIONS, nodes=None):
    """Draw one mode's graph on `ax`; returns the number of stations plotted."""
    G = graphs[rt]['G']
    sub = G.subgraph(nodes) if nodes is not None else G
    pts = {n: (d['lon'], d['lat']) for n, d in sub.nodes(data=True) if has_coords(d)}
    if not pts:
        ax.set_axis_off()
        ax.set_title(f'{MODE_LABELS[rt]} - no usable coordinates')
        return 0
    for u, v, d in sub.edges(data=True):
        if u in pts and v in pts:
            ax.plot([pts[u][0], pts[v][0]], [pts[u][1], pts[v][1]],
                    color=MODE_COLORS.get(rt, '#475569'), linewidth=1.6, alpha=0.75, zorder=1)
    load = np.array([stop_calls[rt].get(n, 0) for n in pts], dtype=float)
    sizes = 18 + 90 * (load / load.max()) if load.max() > 0 else np.full(len(pts), 25.0)
    xs = [p[0] for p in pts.values()]
    ys = [p[1] for p in pts.values()]
    ax.scatter(xs, ys, s=sizes, c=MODE_COLORS.get(rt, '#475569'),
               edgecolors='white', linewidths=0.6, zorder=2)
    if label_stations and len(pts) <= max_labels:
        placed = set()
        for n, (x, y) in pts.items():
            name = sub.nodes[n].get('stop_name', '') or n
            if name in placed:          # both directional platforms carry the same name
                continue
            placed.add(name)
            ax.annotate(name, (x, y), fontsize=7, xytext=(3, 3),
                        textcoords='offset points', zorder=3)
    ax.set_aspect(1 / np.cos(np.deg2rad(float(np.mean(ys)))))
    ax.set_xlabel('Longitude', fontsize=9)
    ax.set_ylabel('Latitude', fontsize=9)
    return len(pts)


fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for ax, rt in zip(axes.ravel(), MINOR_ROUTE_TYPES):
    plotted = draw_mode(ax, rt)
    row = minor_summary.loc[minor_summary['route_type'] == rt].iloc[0]
    ax.set_title(f'{MODE_LABELS[rt]} (route_type {rt})\n'
                 f'{row.graph_nodes} stations, {row.undirected_edges} links, '
                 f'{row.components} component(s), {plotted} plotted', fontsize=10)
fig.suptitle('Each minor mode is a single metropolitan or rural artefact, not a national network',
             fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / 'mode_network_maps.png', dpi=FIG_DPI)
plt.show()

### 14a. קווי הרכבת הקלה, עם שמות תחנות

הרכבת הקלה קטנה דיה כדי לתייג אותה במלואה, וכך מבנה המסדרון נעשה קריא: כל קו הוא שרשרת תחנות ממסוף אחד למשנהו, עם הסתעפות קצרה אחת לכל היותר. שני הפאנלים מפצלים את גרף הרכבת הקלה לפי אזור מטרופוליני (`metro`), משום שהמערכות בירושלים ובתל אביב הן רשתות נפרדות לחלוטין החולקות אך ורק `route_type`. רציפים כיווניים כפולים מתויגים פעם אחת לכל שם, בהתאם לחלק 11.

אותו טיפול ניתן ל-cable tram, ששני רכיביו - הפוניקולר "כרמלית" והרכבלית בחיפה - הם בין ה"רשתות" הקצרות ביותר שניתן לצייר.

In [ ]:
# --- Labelled maps: light rail by metro, then cable tram ------------------
G0 = graphs[0]['G']
groups = defaultdict(list)
for n, d in G0.nodes(data=True):
    groups[d.get('metro') or d.get('region') or 'unknown'].append(n)
groups = dict(sorted(groups.items(), key=lambda kv: -len(kv[1])))

if groups:
    fig, axes = plt.subplots(1, len(groups), figsize=(7.5 * len(groups), 9.5))
    axes = np.atleast_1d(axes)
    for ax, (metro, nodes) in zip(axes, groups.items()):
        plotted = draw_mode(ax, 0, label_stations=True, nodes=nodes)
        sub = G0.subgraph(nodes)
        ncomp = nx.number_connected_components(sub)
        names = len({sub.nodes[n]['stop_name'] for n in nodes})
        ax.set_title(f'Light rail - {metro}\n{len(nodes)} platform stop_ids / ~{names} named '
                     f'stations, {ncomp} graph component(s)', fontsize=11)
    fig.suptitle('Light rail: each line appears twice, once per direction of travel', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES / 'lightrail_labeled_map.png', dpi=FIG_DPI)
    plt.show()

fig, ax = plt.subplots(figsize=(7.5, 8))
draw_mode(ax, 5, label_stations=True)
row5 = minor_summary.loc[minor_summary['route_type'] == 5].iloc[0]
ax.set_title(f'Cable tram (route_type 5) - {row5.graph_nodes} stations in '
             f'{row5.components} separate systems, {row5.trips:,} trips/day', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / 'cabletram_labeled_map.png', dpi=FIG_DPI)
plt.show()

## 15. Betweenness מדויק לאורך כל מודל

מרכזיות betweenness סופרת את המסלולים הקצרים ביותר העוברים דרך תחנה. בגרף מסלול (path) יש לה פרופיל פרבולי אופייני - התחנה האמצעית נמצאת על מרב זוגות המקור-יעד, והמסופים על אף אחד - וכל סטייה מהפרבולה הזו מסמנת צומת או הסתעפות.

הערכים הללו הם **מדויקים** (Brandes, מנורמל, ללא משקלים), ולא הקירוב מבוסס דגימת `k` שנוטבוק 04 נאלץ להשתמש בו על הגרף הארצי בן 30k הצמתים, ואין הסתייגות דגימה הנלווית להם. הגרף מציג את פרופיל ה-betweenness הממוין לכל מודל, והתחנות המובילות מוצגות בטבלה. הסדרים המוחלטים אינם ברי-השוואה למספרים הארציים, משום ש-betweenness מנורמל בתוך כל גרף בנפרד: תחנה יכולה להיות האובייקט המרכזי ביותר בכרמלית ועדיין להיות חסרת חשיבות ארצית.

In [ ]:
# --- Exact betweenness profiles -------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for rt in MINOR_ROUTE_TYPES:
    vals = sorted(BETW[rt].values(), reverse=True)
    if not vals:
        continue
    axes[0].plot(range(1, len(vals) + 1), vals, marker='o', markersize=3,
                 color=MODE_COLORS.get(rt), label=f'{MODE_LABELS[rt]} (n={len(vals)})')
axes[0].set_xlabel('Station rank within its mode')
axes[0].set_ylabel('Exact betweenness (normalised)')
axes[0].set_title('Exact betweenness profile - no sampling error')
axes[0].legend(fontsize=8)

top_bt = (station_metrics.sort_values('exact_betweenness', ascending=False)
          .head(TOP_N).iloc[::-1])
axes[1].barh(range(len(top_bt)),
             top_bt['exact_betweenness'].to_numpy(),
             color=[MODE_COLORS.get(rt, '#475569') for rt in top_bt['route_type']])
axes[1].set_yticks(range(len(top_bt)))
axes[1].set_yticklabels([f'{r.stop_name} [{r.mode_label}]' for r in top_bt.itertuples()],
                        fontsize=8)
axes[1].set_xlabel('Exact betweenness (normalised within mode)')
axes[1].set_title(f'Top {TOP_N} minor-mode stations by exact betweenness')
plt.tight_layout()
plt.savefig(FIGURES / 'exact_betweenness.png', dpi=FIG_DPI)
plt.show()

station_metrics.sort_values('exact_betweenness', ascending=False).head(TOP_N)[
    ['mode_label', 'stop_name', 'metro', 'degree', 'weighted_degree',
     'exact_betweenness', 'is_articulation_point']]

## 16. נזק מדויק מהסרת תחנה בודדת

בגרפים קטנים כל כך איננו זקוקים לסימולציית תקיפה מדגמית: אנו יכולים להסיר **כל** תחנה בתורה ולמדוד את התוצאה המדויקת. שני מדדים נרשמים לכל תחנה:

* `largest_component_share_surviving` - גודל הרכיב השורד הגדול ביותר חלקי `n - 1`. בגרף מסלול, הסרת תחנה סמוך לאמצע חוצה זאת מיד לחצי.
* `pairs_disconnected_share` - שיעור זוגות התחנות שיכלו להגיע זה לזה לפני ההסרה (למעט זוגות המערבים את התחנה שהוסרה עצמה) ואינם יכולים עוד. זהו מדד הנזק הנקי יותר משום שהוא אינו רגיש לשאלה איזה צד של החתך גדול יותר. קו הבסיס מחריג נכונה את הצומת שהוסר: אם הוא ישב ברכיב בגודל `k`, אזי `k - 1` זוגות נעלמים בהגדרה ואינם נספרים כנזק.

העלות היא `O(n)` סריקות קשירות לכל מודל - מאות פעולות בסך הכול. שימו לב מה זה מודד ומה לא: מדובר בבידוד *טופולוגי* בתוך מודל תחבורה יחיד. אין בכך אמירה על יכולתו של נוסע לעבור לאוטובוס, שהיא שאלה רב-מודלית שנוטבוק 17 מטפל בה - וכפי שחלק 17 מראה, שאלה שה-feed הזה עונה עליה בפסימיות, שכן תחנות רכבת קלה אינן חולקות `stop_id` עם אף תחנת אוטובוס.

In [ ]:
# --- Remove every station in turn (exact, no sampling) --------------------
def single_station_damage(G):
    """Exact damage caused by removing each node once."""
    comps = list(nx.connected_components(G))
    size_of = {n: len(c) for c in comps for n in c}
    base_pairs = sum(len(c) * (len(c) - 1) // 2 for c in comps)
    n = G.number_of_nodes()
    out = []
    for node in G.nodes():
        H = G.copy()
        H.remove_node(node)
        after = list(nx.connected_components(H))
        pairs_after = sum(len(c) * (len(c) - 1) // 2 for c in after)
        # pairs that existed among the OTHER nodes before the removal
        baseline = base_pairs - (size_of[node] - 1)
        largest = max((len(c) for c in after), default=0)
        out.append({
            'stop_id': node,
            'largest_component_share_surviving': round(largest / (n - 1), 4) if n > 1 else np.nan,
            'pairs_disconnected_share': round(1 - pairs_after / baseline, 4) if baseline else 0.0,
            'components_after': len(after),
        })
    return pd.DataFrame(out)


damage_frames = []
for rt in MINOR_ROUTE_TYPES:
    G = graphs[rt]['G']
    if G.number_of_nodes() < 2:
        continue
    d = single_station_damage(G)
    d['route_type'] = rt
    d['mode_label'] = MODE_LABELS[rt]
    damage_frames.append(d)

damage = pd.concat(damage_frames, ignore_index=True)
damage = damage.merge(
    station_metrics[['stop_id', 'route_type', 'stop_name', 'metro', 'degree',
                     'weighted_degree', 'exact_betweenness', 'is_articulation_point']],
    on=['stop_id', 'route_type'], how='left')
damage = damage.sort_values(['route_type', 'pairs_disconnected_share'],
                            ascending=[True, False]).reset_index(drop=True)
damage.to_csv(TABLES / 'minor_mode_single_station_damage.csv', index=False, encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(11, 8))
top_dmg = damage.sort_values('pairs_disconnected_share', ascending=False).head(TOP_N).iloc[::-1]
ax.barh(range(len(top_dmg)), top_dmg['pairs_disconnected_share'].to_numpy(),
        color=[MODE_COLORS.get(rt, '#475569') for rt in top_dmg['route_type']])
ax.set_yticks(range(len(top_dmg)))
ax.set_yticklabels([f'{r.stop_name} [{r.mode_label}]' for r in top_dmg.itertuples()], fontsize=8)
ax.set_xlabel('Share of station pairs disconnected by removing this single station')
ax.set_title(f'Exact single-station damage - top {TOP_N} minor-mode stations')
plt.tight_layout()
plt.savefig(FIGURES / 'single_station_damage.png', dpi=FIG_DPI)
plt.show()

print('Mean share of pairs disconnected by removing one station, per mode:')
print(damage.groupby('mode_label')['pairs_disconnected_share']
      .agg(['mean', 'max', 'count']).round(3).to_string())

## 17. היכן ממוקמים המודלים המשניים בתוך הגרף הארצי?

בדיקת הצלבה אופציונלית אך מאירת עיניים, המורצת רק אם נוטבוק 02 ייצר `nodes.csv` ו-`edges.csv`. נוטבוק 03 דיווח שהגרף הארצי אינו קשיר לגמרי: כתריסר רכיבים, שאחד מהם מכיל כ-99.3% מהתחנות. תא זה שואל **מהם הרכיבים האחרים** על ידי תיוג כל רכיב ארצי במודלים המשרתים את תחנותיו.

התשובה הצפויה - וב-feed הזה, בפועל - היא שהרכיבים הקטנים הם בדיוק המודלים המשניים בתוספת הרכבת. תחנות רכבת קלה, cable tram ומוניות שירות אינן חולקות **אף** `stop_id` עם תחנת אוטובוס כלשהי, ולכן בגרף שצמתיו הם `stop_id` הן אינן יכולות לגעת ברשת האוטובוסים כלל: נוסע העובר מחשמלית לאוטובוס הולך בין שתי תחנות שהמודל מתייחס אליהן כבלתי קשורות. שירותי ההיענות לביקוש הם היוצא מן הכלל - רוב תחנותיהם הן תחנות אוטובוס רגילות, ולכן הם נבלעים לתוך הרכיב הענק.

שתי השלכות שראוי לומר במפורש. ראשית, ה"פיצול" של הרשת הארצית שדווח בנוטבוק 03 הוא בעיקרו אפקט של *קידוד* מזהי תחנה נפרדים, ולא עדות למערכת תחבורה מנותקת. שנית, מספרי הנזק של נוטבוק זה הם חסם תחתון על החוסן במציאות וחסם עליון על הקישוריות המודלית: בתוך מודל תחבורה הם מדויקים, אך שום תחלופה בין מודלים אינה מיוצגת במודל בשום מקום.

In [ ]:
# --- Optional: locate the minor modes in the national graph ---------------
if nodes_path is None or edges_path is None:
    print('notebook 02 artifacts not found - national placement cross-check skipped '
          '(run 02_graph_construction to enable it).')
else:
    nat_edges = pd.read_csv(edges_path, dtype={'from_stop': str, 'to_stop': str},
                            encoding='utf-8-sig')
    NAT = nx.from_pandas_edgelist(nat_edges, 'from_stop', 'to_stop')
    nat_comps = sorted(nx.connected_components(NAT), key=len, reverse=True)

    stop_to_modes = defaultdict(set)
    for rt, stops in mode_stops.items():
        for s in stops:
            stop_to_modes[s].add(rt)

    rows = []
    for i, c in enumerate(nat_comps):
        counts = Counter()
        for s in c:
            for rt in stop_to_modes.get(s, ()):
                counts[rt] += 1
        rows.append({
            'component_rank': i,
            'nodes': len(c),
            'modes_present': ', '.join(f'{MODE_LABELS.get(rt, rt)}:{v}'
                                       for rt, v in counts.most_common()),
            'dominant_mode': MODE_LABELS.get(counts.most_common(1)[0][0], '?') if counts else '?',
            'sample_station': sorted(ATTR.get(s, DEFAULT_ATTR)['stop_name'] for s in c)[0],
        })
    nat_comp_df = pd.DataFrame(rows)
    nat_comp_df.to_csv(TABLES / 'national_component_modes.csv', index=False, encoding='utf-8-sig')

    print(f'national graph: {NAT.number_of_nodes():,} nodes, {len(nat_comps)} components')
    display(nat_comp_df)

    # How many stops of each minor mode are shared with another mode at all?
    print('Stops shared with at least one other mode (same stop_id):')
    for rt in MINOR_ROUTE_TYPES:
        stops = mode_stops.get(rt, set())
        shared = sum(1 for s in stops if len(stop_to_modes[s]) > 1)
        print(f'  {MODE_LABELS[rt]:<18} {shared:>4} / {len(stops):<4} '
              f'({shared / max(len(stops), 1):.0%}) - '
              f'{"integrated with the bus network" if shared else "no shared stop_id with any other mode"}')

## 18. מתי המודלים הללו פועלים בפועל?

הממד האחרון של עצימות הוא הזמן. באמצעות ערכי השניות-מאז-חצות-יום-השירות שפוענחו בחלק 7, אנו סופרים יציאות לכל שעת שירות עבור כל מודל. השעות 24, 25, ... נשמרות כסלים נפרדים ואינן מקופלות בחזרה ל-0, 1, ... - אלה שירותי לילה מאוחר של יום השירות ה*קודם*, ומיזוגם היה גם מעוות את השעות הקטנות וגם מסתיר את עצם העובדה שהמודלים הללו פועלים אחרי חצות.

הפרופיל מפריד בין מודלים של תחבורה המונית אמיתית לבין השאר: מודל בעל פרופיל חלק לאורך כל היום, טווח ארוך ומרווח שירות של דקות בודדות נושא מסדרון מטרופוליני; מודל בעל כמה עשרות יציאות המרוכזות בשעות המשרד הוא שירות נישה, שאין להפריז בפרשנות סטטיסטיקות ה"רשת" שלו.

In [ ]:
# --- Departures per service hour ------------------------------------------
hour_rows = []
for rt in MINOR_ROUTE_TYPES:
    deps = [d for lst in stop_departures[rt].values() for d in lst]
    counts = Counter(d // 3600 for d in deps)
    for h in range(0, 28):
        hour_rows.append({'route_type': rt, 'mode_label': MODE_LABELS[rt],
                          'service_hour': h, 'departures': counts.get(h, 0)})
hourly = pd.DataFrame(hour_rows)
hourly.to_csv(TABLES / 'hourly_departures.csv', index=False, encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(12, 5.5))
for rt in MINOR_ROUTE_TYPES:
    d = hourly[hourly['route_type'] == rt]
    total = d['departures'].sum()
    if total == 0:
        continue
    ax.plot(d['service_hour'], d['departures'] / total, marker='o', markersize=3,
            color=MODE_COLORS.get(rt), label=f'{MODE_LABELS[rt]} ({total:,} calls)')
ax.axvline(24, color='#111827', linestyle='--', linewidth=1)
ax.text(24.1, ax.get_ylim()[1] * 0.9, 'GTFS hours >= 24\n(after midnight)', fontsize=8)
ax.set_xlabel('Service hour (seconds since service midnight // 3600)')
ax.set_ylabel('Share of the mode\'s stop calls')
ax.set_title('When each minor mode runs')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / 'hourly_departures.png', dpi=FIG_DPI)
plt.show()

minor_summary[['mode_label', 'first_departure', 'last_departure', 'service_span_hours',
               'departures_after_midnight', 'median_headway_busiest_stop_minutes',
               'max_calls_at_a_stop']]

## 19. שמירת סיכום השלב

המספרים המרכזיים נאספים אל `lightrail_summary.json` (ארטיפקט החוזה של שלב זה), ארבעת הגרפים נשמרים ב-pickle כ-`{mode_label: networkx.Graph}` כך שנוטבוק מאוחר יותר יוכל לעשות בהם שימוש חוזר בלי לזרום מחדש על ה-feed בן 816 MB, וכל קובץ שנכתב על ידי נוטבוק זה מוצג עם גודלו כבדיקה סופית לכך ששום דבר לא נחת מחוץ לתיקיית השלב.

In [ ]:
# --- Stage summary and artifacts ------------------------------------------
lr = minor_summary.loc[minor_summary['route_type'] == 0].iloc[0]
bus_row = intensity.loc[intensity['route_type'] == 3]
bus_tpr = float(bus_row['trips_per_route'].iloc[0]) if len(bus_row) else None

summary = {
    'modes_analysed': {str(rt): MODE_LABELS[rt] for rt in MINOR_ROUTE_TYPES},
    'exact_centrality': bool(EXACT_BETWEENNESS),
    'stream_stats': stream_stats,
    'lightrail': {
        'routes': int(lr.routes), 'trips': int(lr.trips),
        'stop_ids': int(lr.graph_nodes),
        'distinct_station_names': int(platform_dup.loc[
            platform_dup['route_type'] == 0, 'distinct_stop_names'].iloc[0]),
        'undirected_edges': int(lr.undirected_edges),
        'components': int(lr.components),
        'cyclomatic_number': int(lr.cyclomatic_number),
        'articulation_points': int(lr.articulation_points),
        'articulation_point_share': float(lr.articulation_point_share),
        'bridges': int(lr.bridges),
        'trips_per_route': float(lr.trips_per_route),
        'max_calls_at_a_stop': int(lr.max_calls_at_a_stop),
        'median_headway_busiest_stop_minutes': (
            None if pd.isna(lr.median_headway_busiest_stop_minutes)
            else float(lr.median_headway_busiest_stop_minutes)),
        'service_span_hours': float(lr.service_span_hours),
        'shape_class': lr.shape_class,
    },
    'bus_trips_per_route_baseline': bus_tpr,
    'per_mode': {
        MODE_LABELS[rt]: {
            k: (None if pd.isna(v) else (v.item() if hasattr(v, 'item') else v))
            for k, v in minor_summary.loc[
                minor_summary['route_type'] == rt].iloc[0].to_dict().items()
        }
        for rt in MINOR_ROUTE_TYPES
    },
    'caveats': [
        'route_type 8 is labelled trolleybus in GTFS but is operated by shared-taxi companies.',
        'Light-rail directional platforms are separate stop_ids, so each line appears as two '
        'one-way components; component counts are partly a publishing convention.',
        'Weights are scheduled trips, not passengers - GTFS carries no ridership data.',
        'Damage measures are within-mode only; no modal substitution is represented.',
    ],
}
with open(STAGE / 'lightrail_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)

with open(STAGE / 'lightrail_graphs.pkl', 'wb') as f:
    pickle.dump({MODE_LABELS[rt]: graphs[rt]['G'] for rt in MINOR_ROUTE_TYPES}, f)

print('Artifacts written under', STAGE)
for p in sorted(STAGE.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(STAGE)}  ({p.stat().st_size / 1024:,.1f} KB)')

## מסקנות

*(המספרים שלהלן הם אלו שהודפסו על ידי התאים לעיל על תצלום ה-feed הזה; הריצו מחדש כדי לרעננם.)*

* **המודלים המשניים הם 0.8% מהקווים ו-1.5% מהנסיעות, אך אינם קטנים בעצימותם.** ה-cable tram מפעיל כ-752 נסיעות לקו והרכבת הקלה כ-361, לעומת כ-61 באוטובוס וכ-1.2 ברכבת. במדידה לפי קו, אלה השירותים המנוצלים ביותר בארץ: רציף רכבת קלה יחיד בתל אביב רואה למעלה מ-1,200 עצירות מתוזמנות ביום שירות, כלומר חשמלית בערך כל 70 שניות לאורך טווח ההפעלה, ותחנות הרכבלית בחיפה רואות כ-2,400 עצירות כל אחת.
* **אלה מסדרונות, לא רשתות.** גרף הרכבת הקלה הוא **יער** - מספר ציקלומטי 0 - ולכן *כל* קשת היא bridge וכ-93% מתחנותיו הן articulation points; ה-cable tram הוא באופן דומה שני מסלולים זרים. השוו למספרים הארציים של נוטבוק 03: כ-3% מהתחנות הן articulation points ופחות מ-2% מהקשתות הן bridges. זוהי הנקודה המבנית של הנוטבוק: על מסלול אין מסלול חלופי מעצם הבנייה, ולכן היתירות המאפשרת לרשת האוטובוסים לספוג כשלים פשוט אינה קיימת כאן.
* **לפיכך יש למדוד קריטיות לפי *כמה* נותר מנותק, ולא לפי *האם* משהו נותר מנותק.** הסרות התחנה הבודדות המדויקות בחלק 16 מראות שהנזק עולה בהדרגה לעבר אמצע כל מסדרון - הסרת תחנה באמצע קו מנתקת בערך מחצית מכלל זוגות התחנות באותו קו - בעוד שתחנות מסוף עולות כמעט כלום. בשילוב עם נתוני העומס, תחנות הרכבת הקלה שבאמצע המסדרון הן נקודות הכשל הבודדות בעלות ההשלכות החמורות ביותר בכל קבוצת המודלים המשניים: תנועה גבוהה, יתירות אפסית.
* **המודל המשני היחיד שהוא meshed באמת הוא שירות ההיענות לביקוש (route_type 715).** יש לו 153 תחנות, כ-194 קשתות ומספר ציקלומטי של כ-43 - רשת ממוששת אמיתית עם מסלולים חלופיים - משום שווריאנטים שונים של נסיעה באותו קו כפרי עוברים בכבישים שונים. זהו ארטיפקט של אופן פרסום הניתוב הגמיש, ולא עדות למערכת פיזית יתירה, ואין לקרוא זאת כ"שירות כפרי הוא בעל חוסן רב יותר".
* **שתי בעיות תיוג הן מהותיות ואין להתעלם מהן.** (1) `route_type = 8` הוא "trolleybus" במפרט ה-GTFS אך מופעל כאן על ידי חברות מוניות - 8 קווים, 47 נסיעות, ולכל נסיעה יש בדיוק שתי עצירות תחנה, כך שה"גרף" שלו הוא שלושה גדמים מנותקים ללא כל פנים. אין להסיק ממנו שום מסקנה רשתית. (2) כל קו רכבת קלה מפורסם כשני `route_id` כיווניים עם **`stop_id` נפרדים לרציפים**, וזו הסיבה לכך ש-138 stop_ids של רכבת קלה מתאימים לכ-69 תחנות נקובות בלבד, ולכך שהמודל מציג 4 רכיבים במקום 2 קווים. יש לקרוא כל נתון לכל `stop_id` ברכבת הקלה עם מקדם שתיים זה בחשבון.
* **המודלים המשניים הם בדיוק הפיצול שהניתוח הארצי דיווח עליו.** כל רכיב קטן של הגרף הארצי, מלבד הרכבת ואשכול אוטובוסים תועה אחד, הוא מודל משני: רכבת קלה, cable tram ומוניות שירות אינן חולקות `stop_id` עם אף תחנת אוטובוס, ולכן במודל מבוסס stop-id הן אינן יכולות לגעת ברשת האוטובוסים. "כתריסר רכיבים" של נוטבוק 03 הוא אפוא בעיקרו אפקט קידוד, וכל ניתוח מעברים רב-מודלי (נוטבוק 17) חייב להתבסס על קרבה מרחבית ולא על מזהים משותפים, אחרת יסיק בטעות שלא קיימות נקודות מעבר כלל.
* **מגבלות, בכנות.** המשקלים הם נסיעות מתוזמנות, לעולם לא נוסעים - ל-GTFS אין נתוני נסועה, ולכן "עומס" כאן משמעו *היצע*. הנזק נמדד בתוך מודל תחבורה יחיד, ולכן כשל ברכבת קלה שקו אוטובוס מקביל היה סופג נספר בכל זאת כניתוק מלא. ה-feed הוא תצלום יחיד: רק קווי הרכבת הקלה הפועלים בו מופיעים, וקווים שנפתחו מאוחר יותר אינם קיימים במספרים אלה. לבסוף, betweenness מנורמל בתוך כל גרף מודל בנפרד, ולכן ערך של 0.2 בכרמלית וערך של 0.2 ברמה הארצית אינם אותו גודל ואין להשוות ביניהם בין טבלאות.